In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
print(train.shape)  # (1460, 81)
print(test.shape)   # (1459, 80)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
print("Import xong")

In [ ]:

train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
print(f"Train: {train.shape}")  # (1460, 81)
print(f"Test:  {test.shape}")   # (1459, 80)

In [ ]:

# Xem 5 dòng đầu
print("=== 5 dòng đầu ===")
display(train.head())

In [ ]:
# Xem thống kê cơ bản
print("\n=== Thống kê ===")
display(train.describe())

In [ ]:
# Xem phân phối giá nhà
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

In [ ]:
train['SalePrice'].hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice gốc — lệch phải')
axes[0].set_xlabel('Giá (USD)')

In [ ]:
np.log1p(train['SalePrice']).hist(bins=50, ax=axes[1], color='green')
axes[1].set_title('log1p(SalePrice) — chuẩn hơn')
axes[1].set_xlabel('log(Giá)')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:

# Đếm missing values và tính %
missing = train.isnull().sum() # Đếm số giá trị thiếu
missing_pct = (missing / len(train) * 100).round(1) # tính %
# gộp lại thành 1 bảng
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

In [ ]:
print(f"Số cột có missing: {len(missing_df)}")
display(missing_df.head(20))

In [ ]:

# Tách giá nhà ra, log transform
y = np.log1p(train['SalePrice']) # Tách biến mục tiêu (target) + log transform
print(f"y shape: {y.shape}")
print(f"y range: {y.min():.2f} → {y.max():.2f}  (log scale)") # Đây là giá sau khi log, không phải USD thật
print(f"Tương đương: ${np.expm1(y.min()):,.0f} → ${np.expm1(y.max()):,.0f}")

In [ ]:
# Gộp train + test để xử lý missing values và encoding đồng nhất
all_data = pd.concat([
    train.drop(['SalePrice', 'Id'], axis=1), # bỏ SalePrice vì đã tách ra y, bỏ id vì không có ý nghĩa cho model
    test.drop(['Id'], axis=1)
], axis=0, ignore_index=True)

In [ ]:
print(f"\nAll data shape: {all_data.shape}")  # (2919, 79)

In [ ]:

# ── NHÓM 1: Trống = "Không có" ──────────────────────────────
# Những cột này trống vì nhà không có feature đó (không phải lỗi data)
none_fill_cols = [
    'PoolQC',        # Không có hồ bơi
    'MiscFeature',   # Không có tính năng đặc biệt
    'Alley',         # Không có hẻm
    'Fence',         # Không có hàng rào
    'FireplaceQu',   # Không có lò sưởi
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType',
]
for col in none_fill_cols:
    all_data[col] = all_data[col].fillna('None')

In [ ]:
# ── NHÓM 2: Trống = 0 ────────────────────────────────────────
# Những cột số mà trống vì không có feature tương ứng
zero_fill_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea',
]
for col in zero_fill_cols:
    all_data[col] = all_data[col].fillna(0)

In [ ]:
# ── NHÓM 3: LotFrontage — điền median theo khu phố ──────────
# Chiều dài mặt tiền thường tương đồng trong cùng khu phố
all_data['LotFrontage'] = (
    all_data.groupby('Neighborhood')['LotFrontage']
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
# ── NHÓM 4: Categorical còn lại — điền mode (giá trị phổ biến nhất) ──
cat_cols = all_data.select_dtypes(include='object').columns
for col in cat_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

In [ ]:
# ── NHÓM 5: Numeric còn lại — điền median ───────────────────
num_cols = all_data.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    all_data[col] = all_data[col].fillna(all_data[col].median())

In [ ]:
# Kiểm tra
remaining = all_data.isnull().sum().sum()
print(f"Missing còn lại: {remaining}")  # → 0

In [ ]:

# Tổng diện tích sử dụng — feature quan trọng nhất
all_data['TotalSF'] = (
    all_data['TotalBsmtSF'] +
    all_data['1stFlrSF'] +
    all_data['2ndFlrSF']
)

In [ ]:
# Tổng số phòng tắm (full bath = 1 điểm, half bath = 0.5 điểm)
all_data['TotalBath'] = (
    all_data['FullBath'] +
    all_data['BsmtFullBath'] +
    0.5 * all_data['HalfBath'] +
    0.5 * all_data['BsmtHalfBath']
)

In [ ]:
# Tuổi nhà tại thời điểm bán
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']

In [ ]:
# Số năm kể từ lần sửa chữa gần nhất
all_data['RemodelAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

In [ ]:
# Nhà có được sửa chữa không? (1 = có, 0 = không)
all_data['WasRemodeled'] = (
    all_data['YearRemodAdd'] != all_data['YearBuilt']
).astype(int)

In [ ]:
# Nhà mới xây? (bán ngay năm xây hoặc năm sau)
all_data['IsNew'] = (
    all_data['YrSold'] - all_data['YearBuilt'] <= 1
).astype(int)

In [ ]:
# Có garage / tầng hầm / hồ bơi không?
all_data['HasGarage']   = (all_data['GarageArea']   > 0).astype(int)
all_data['HasBasement'] = (all_data['TotalBsmtSF']  > 0).astype(int)
all_data['HasPool']     = (all_data['PoolArea']      > 0).astype(int)

In [ ]:
print(f"Số features sau khi thêm: {all_data.shape[1]}")  # ~87

In [ ]:
# Xem feature mới có tương quan tốt không
new_features = ['TotalSF', 'TotalBath', 'HouseAge', 'RemodelAge']
correlations = []
train_idx = range(len(train))
for f in new_features:
    corr = all_data.iloc[train_idx][f].corr(y) # lấy cột feature chỉ trong tập train
    correlations.append((f, round(corr, 3)))
    print(f"  {f:15s} → corr với SalePrice: {corr:.3f}")

In [ ]:

# Encode categorical features
# Tất cả cột text cần chuyển thành số vì model chỉ hiểu số
cat_cols = all_data.select_dtypes(include='object').columns.tolist()
print(f"Số cột cần encode: {len(cat_cols)}")

In [ ]:
# Biến mỗi giá trị text thành số nguyên
le = LabelEncoder() # Tạo encoder
for col in cat_cols:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

In [ ]:
# Kiểm tra — tất cả phải là số
print(f"\nKiểu dữ liệu còn lại: {all_data.dtypes.unique()}")
print(f"Shape cuối: {all_data.shape}")

In [ ]:

# Tách lại train/test & chia validation
n_train = len(train)

In [ ]:
X_full     = all_data.iloc[:n_train, :]   # Dùng để train cuối cùng
X_test_sub = all_data.iloc[n_train:, :]   # Dùng để submit Kaggle (nếu muốn)

In [ ]:
# Chia train → train + validation để đánh giá model
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y,
    test_size=0.2, # train 80%, test 20%
    random_state=42
)

In [ ]:
print(f"X_train: {X_train.shape}")  # (1168, ~87)
print(f"X_val:   {X_val.shape}")    # (292, ~87)
print(f"y_train range: {y_train.min():.2f} → {y_train.max():.2f}")

In [ ]:

xgb = XGBRegressor(
    n_estimators     = 1000,   # Số cây tối đa — early stopping sẽ dừng sớm hơn
    learning_rate    = 0.05,   # Mỗi cây đóng góp 5% — học chậm nhưng chắc
    max_depth        = 4,      # Mỗi cây sâu tối đa 4 tầng — tránh overfit
    subsample        = 0.8,    # Mỗi cây chỉ dùng 80% dòng data ngẫu nhiên
    colsample_bytree = 0.8,    # Mỗi cây chỉ dùng 80% cột ngẫu nhiên
    reg_alpha        = 0.1,    # L1 regularization — phạt features không cần thiết
    reg_lambda       = 1.0,    # L2 regularization — tránh weight quá lớn
    random_state     = 42,
    eval_metric      = 'rmse',
    early_stopping_rounds = 50  # Dừng nếu 50 cây liên tiếp không cải thiện
)

In [ ]:
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100   # In kết quả mỗi 100 cây
)

In [ ]:
print(f"\n Dừng ở cây thứ: {xgb.best_iteration}")

In [ ]:

import lightgbm as lgb_module

In [ ]:
lgb = LGBMRegressor(
    n_estimators     = 1000,
    learning_rate    = 0.05,
    max_depth        = 4,
    num_leaves       = 31,    # LightGBM dùng num_leaves thay max_depth để kiểm soát độ phức tạp
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    random_state     = 42,
)

In [ ]:
lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb_module.early_stopping(50),
        lgb_module.log_evaluation(100)
    ]
)

In [ ]:
print(f"\n Dừng ở cây thứ: {lgb.best_iteration_}")

In [ ]:

# Predict trên validation set
pred_xgb = xgb.predict(X_val)
pred_lgb = lgb.predict(X_val)
pred_ensemble = 0.5 * pred_xgb + 0.5 * pred_lgb

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{name:15s} → RMSE: {rmse:.4f}  |  R²: {r2:.4f}")

In [ ]:
print("=== Kết quả trên Validation Set (log scale) ===")
evaluate("XGBoost",   y_val, pred_xgb)
evaluate("LightGBM",  y_val, pred_lgb)
evaluate("Ensemble",  y_val, pred_ensemble)

In [ ]:
# Xem sai số thực tế bằng USD
price_true = np.expm1(y_val)
price_pred = np.expm1(pred_ensemble)
mae_usd = np.abs(price_true - price_pred).mean()
print(f"\nSai số trung bình (USD): ${mae_usd:,.0f}")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
# Plot 1: Predicted vs Actual
axes[0].scatter(np.expm1(y_val), np.expm1(pred_ensemble),
                alpha=0.4, color='steelblue', s=20)
max_val = max(np.expm1(y_val).max(), np.expm1(pred_ensemble).max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Giá thực tế (USD)')
axes[0].set_ylabel('Giá dự đoán (USD)')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

In [ ]:
# Plot 2: Top 15 features quan trọng nhất
feat_importance = pd.Series(
    xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True).tail(15)

In [ ]:
feat_importance.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Features quan trọng nhất (XGBoost)')
axes[1].set_xlabel('Importance score')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:

feature_names = list(X_train.columns)

In [ ]:
joblib.dump(xgb,           '/kaggle/working/xgboost_model.pkl')
joblib.dump(lgb,           '/kaggle/working/lightgbm_model.pkl')
joblib.dump(feature_names, '/kaggle/working/feature_names.pkl')

In [ ]:
# Verify — load lại và predict thử
xgb_check = joblib.load('/kaggle/working/xgboost_model.pkl')
lgb_check  = joblib.load('/kaggle/working/lightgbm_model.pkl')
features_check = joblib.load('/kaggle/working/feature_names.pkl')

In [ ]:
sample = X_val.iloc[[0]][features_check]
p = 0.5 * xgb_check.predict(sample) + 0.5 * lgb_check.predict(sample)
print(f"Verify OK — Dự đoán: ${np.expm1(p[0]):,.0f}")
print(f"   Thực tế:            ${np.expm1(y_val.iloc[0]):,.0f}")
print(f"\nSố features: {len(features_check)}")
print(f"Tên features: {features_check[:5]}...")

In [ ]:

# Nhìn sang panel Output bên phải của Kaggle Notebook
# Click icon ⬇ cạnh từng file để download

In [ ]:
# Hoặc chạy cell này để xác nhận file đã có
import os
for f in ['xgboost_model.pkl', 'lightgbm_model.pkl', 'feature_names.pkl']:
    path = f'/kaggle/working/{f}'
    size = os.path.getsize(path) / 1024 / 1024
    print(f"{f:30s} {size:.1f} MB")
